In [ ]:
import sys
sys.path.append('..')

In [ ]:
# from utils.dataloaders import create_dataloader
# from utils.general import (LOGGER, TQDM_BAR_FORMAT, Profile, check_dataset, check_img_size, check_requirements,
#                            check_yaml, coco80_to_coco91_class, colorstr, increment_path, non_max_suppression,
#                            print_args, scale_boxes, xywh2xyxy, xyxy2xywh)

In [ ]:
# task = 'val'
# dataset_yaml = "../datasets/T16bit_25k_09-03-2023/dataset.yaml"
# #dataset_yaml = "../datasets/T8bit_minmax_25k_05-03-2023/dataset.yaml"
# data = check_dataset(dataset_yaml)  # check

# dataloader, dataset = create_dataloader(
#     data[task], # 
#     imgsz=640,
#     batch_size=16,
#     stride=1,
#     single_cls=False,
#     pad=0.5,
#     rect=False,
#     augment=False,
#     prefix=colorstr(f'{task}: '))

# import numpy as np
# import cv2
# from PIL import Image

# for img, labels, im_file, ((h0, w0), (_, pad)) in dataset:
#     print(img.shape, labels.shape, h0, w0, pad)
#     break

# Image.fromarray(img[0].numpy()) #.transpose(1,2,0).astype(np.uint8))

In [ ]:
import numpy as np
import cv2
from PIL import Image

im_fpath = '/mnt/fiftyoneDB/Database/Image_Data/Thermal_Images_16Bit/Trip_310_Seq_153/23714953_r.png'
#im_fpath = '/mnt/fiftyoneDB/Database/Image_Data/Thermal_Images_16Bit/Trip_272_Seq_39/22837305_l.png'
#im_fpath = '/mnt/fiftyoneDB/Database/Image_Data/Thermal_Images_16Bit/Trip_284_Seq_154/23398206_l.png'
im = cv2.imread(im_fpath, cv2.IMREAD_UNCHANGED)
print(f"{im.dtype=}, {np.ma.minimum_fill_value(im)=}, {np.ma.maximum_fill_value(im)=}")
Image.fromarray(im)

In [ ]:
import plotly.express as px

hist_size = 2**16
hist = cv2.calcHist([im], [0], None, [hist_size], [0, 65536])
px.line(x=np.arange(hist_size), y=hist.flatten()).show()

In [ ]:
from PIL import Image
import albumentations as A
from utils.albumentations16 import CLAHE, Clip, NormalizeMinMax

def get_clip_bounds(im, q_lb=0.005, q_ub=0.995):
    hist = cv2.calcHist([im], [0], None, [hist_size], [0, 65536])
    cumsum = np.cumsum(hist)
    cumsum = cumsum / cumsum[-1]
    q_lb = np.argmax(cumsum > q_lb)
    q_ub = np.argmax(cumsum > q_ub)

    q_lb *= round(np.iinfo(im.dtype).max / hist_size)
    q_ub *= round(np.iinfo(im.dtype).max / hist_size)
    print(f"{q_lb=}, {q_ub=}")
    return q_lb, q_ub

def apply_transform(im, llimit, ulimit, clahe_p=0.0):
    transform = A.Compose([
                Clip(p=1.0, lower_limit=(llimit,)*2, upper_limit=(ulimit,)*2),
                CLAHE(p=clahe_p, clip_limit=(4, 4), tile_grid_size=(0, 0)),
                NormalizeMinMax(p=1.0),
                A.UnsharpMask(p=1.0),
                A.ToRGB(p=1.0),
            ])
    return transform(image=im)['image']

q_lb, q_ub = get_clip_bounds(im, q_lb=0.002, q_ub=0.998)
title = f"Clip bounds: {q_lb=}, {q_ub=}"
px.line(x=np.arange(hist_size), y=hist.flatten(), range_x=[q_lb, q_ub], title=title).show()

llimit = 15000 / 65535
ulimit = 28000 / 65535
img_1 = apply_transform(im, llimit, ulimit)

llimit = q_lb / 65535
ulimit = q_ub / 65535
img_2 = apply_transform(im, llimit, ulimit)
img_3 = apply_transform(im, llimit, ulimit, clahe_p=1.0)

Image.fromarray(np.hstack([img_1, img_2, img_3]))

In [ ]:
from utils import albumextensions as Ax

A.GaussNoise(p=1, var_limit=(1, 5), per_channel=False)

ir_blur = Ax.ThermalHorizontalMotionBlur(p=1, tau_range=(6, 6), noise_std_range=(0.01, 0.02))
Image.fromarray(ir_blur(image=img_1)["image"])

In [ ]:
import matplotlib.pyplot as plt
from utils.albumentations16 import convert_16bit_to_8bit

ims = [convert_16bit_to_8bit(im, augment=True) for _ in range(10)]
diffs = np.diff(np.array(ims), axis=0)
print(diffs.mean(axis=(1,2,3)))
# plot images in 4x4 grid
fig, axs = plt.subplots(2, 5, figsize=(16, 8))
for i, ax in enumerate(axs.flatten()):
    ax.imshow(ims[i], cmap='gray')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import cv2

im_fpath = '/mnt/fiftyoneDB/Database/Image_Data/Thermal_Images_16Bit/Trip_295_Seq_39/23590222_l.png'
#im_fpath = '/mnt/fiftyoneDB/Database/Image_Data/Thermal_Images_16Bit/Trip_310_Seq_153/23714953_r.png'

# low contrast imgs
#im_fpath = '/mnt/fiftyoneDB/Database/Image_Data/Thermal_Images_16Bit/Trip_161_NoiseSeq_1/4378236_l.png'
#im_fpath = '/mnt/fiftyoneDB/Database/Image_Data/Thermal_Images_16Bit/Trip_161_NoiseSeq_38/4325708_l.png'
#im_fpath = '/mnt/fiftyoneDB/Database/Image_Data/Thermal_Images_8Bit/Trip_161_NoiseSeq_38/4325419_l.jpg'
#im_fpath = '/mnt/fiftyoneDB/Database/Image_Data/Thermal_Images_8Bit/Trip_259_INUSeq_InitiatVG4/19695306_l.jpg'

im_fpath = im_fpath.replace('8Bit', '16Bit').replace('.jpg', '.png')
im = cv2.imread(im_fpath, cv2.IMREAD_UNCHANGED)
print(f"{im.dtype=}, {im.min()=}, {im.max()=}, {im.max()-im.min()=}")

In [ ]:
import sys
sys.path.append('..')

from PIL import Image
import albumentations as A
from utils.albumentations16 import CLAHE, Clip, NormalizeMinMax

ksize = 7
sigma = 0.3 * ((ksize-1)*0.5 - 1) + 0.8
llimit = 15000 / 65535
ulimit = 28000 / 65535

transform = A.Compose([
            Clip(p=1.0, lower_limit=(llimit,)*2, upper_limit=(ulimit,)*2),
            CLAHE(p=1, clip_limit=(4, 4), tile_grid_size=(0, 0)),
            NormalizeMinMax(p=1.0),
            A.ToRGB(p=1.0),
        ])

image = transform(image=im)['image']

transform = A.Compose([
            Clip(p=1.0, lower_limit=(llimit,)*2, upper_limit=(ulimit,)*2),
            CLAHE(p=1, clip_limit=(4, 4), tile_grid_size=(0, 0)),
            NormalizeMinMax(p=1.0),
            A.UnsharpMask(p=1.0, threshold=5), #, blur_limit=(ksize, ksize), alpha=(0.5, 0.5), sigma_limit=(sigma, sigma)),
            A.ToRGB(p=1.0),
        ])

sharp = transform(image=im)['image']

images = np.concatenate((image, sharp, image-sharp), axis=1)
print(f"{(image - sharp).mean()=}")

Image.fromarray(images)

In [ ]:
from utils.albumextensions import ResizeIfNeeded

ResizeIfNeeded(max_size=639)(image=im)['image'].shape